In [13]:
import numpy as np
import time

In [15]:
#CPU code
# Create Host arrays
N=10000000
a=np.arange(N,dtype=int)
b=np.arange(N,dtype=int)
c=np.zeros(N,dtype=int)

start=time.perf_counter()
c=a+b #numpy's vectorized operation
# for i in range(len(a)):
#   c[i]=a[i]+b[i]
end=time.perf_counter()
print(c)


[       0        2        4 ... 19999994 19999996 19999998]


In [16]:
#calculate the execution time on cpu
print("CPU execution time : ",end-start, "seconds")

CPU execution time :  0.03284467100002075 seconds


In [8]:
from numba import cuda
import time

In [9]:
#gpu Kernel : runs on GPU
@cuda.jit
def vector_addition_kernel(a,b,c):
  i=cuda.grid(1)
  if i<len(a):
    c[i]=a[i]+b[i]
  else:
    c[i]=0


In [10]:
#GPU code
#input vectors
N=10000000
a_cpu=np.arange(N,dtype=int)
b_cpu=np.arange(N,dtype=int)

#output vector
c_cpu=np.zeros(N,dtype=int)

start = time.perf_counter()
#copy vectors to the GPu device
d_a=cuda.to_device(a_cpu)
d_b=cuda.to_device(b_cpu)
d_c=cuda.device_array_like(c_cpu)

#launch gpu configuration
threads_per_block=32
blocks_per_grid=(N+threads_per_block-1)//threads_per_block

#launch GPU kernel
vector_addition_kernel[blocks_per_grid,threads_per_block](d_a,d_b,d_c)

cuda.synchronize()

#copy result back from gpu to cpu
c_cpu=d_c.copy_to_host()
end = time.perf_counter()

print("GPU execution time : ",end-start, "seconds")

print("A:",a_cpu)
print("B:",b_cpu)
print("C:",c_cpu)


GPU execution time :  0.1406775249997736 seconds
A: [      0       1       2 ... 9999997 9999998 9999999]
B: [      0       1       2 ... 9999997 9999998 9999999]
C: [       0        2        4 ... 19999994 19999996 19999998]
